# Week 4 — Proxy-NPS from sentiment/star-rating split

Neither entity publishes an official NPS, so we derive a proxy using the standard
promoter/detractor formula applied to app-store star ratings, pooled across Google Play
and the App Store:

`proxy-NPS = %(rating >= 4) - %(rating <= 2), scaled to -100..100`

This mirrors how NPS is conventionally proxy-derived from review data when a real survey
isn't available (rating 5/4 = promoters, 3 = passive, 2/1 = detractors).

**Input:** `data/processed/reviews_sentiment.csv`

**Output:** `data/processed/proxy_nps.csv`

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

ENTITY_NAMES = {"jupiter": "Jupiter", "hdfc_bank": "HDFC Bank"}

df = pd.read_csv(PROCESSED / "reviews_sentiment.csv")
df["entity"] = df["entity"].map(ENTITY_NAMES)
df["segment"] = pd.cut(
    df["rating"], bins=[0, 2, 3, 5], labels=["detractor", "passive", "promoter"]
)
df.groupby(["entity", "segment"], observed=True).size().unstack(fill_value=0)

segment,detractor,passive,promoter
entity,,,
HDFC Bank,314,35,276
Jupiter,506,23,148


In [2]:
def proxy_nps(group):
    n = len(group)
    pct_promoter = (group["rating"] >= 4).sum() / n * 100
    pct_detractor = (group["rating"] <= 2).sum() / n * 100
    return pd.Series({
        "n_reviews": n,
        "pct_promoter": round(pct_promoter, 1),
        "pct_passive": round(100 - pct_promoter - pct_detractor, 1),
        "pct_detractor": round(pct_detractor, 1),
        "proxy_nps": round(pct_promoter - pct_detractor, 1),
    })

nps_by_entity = df.groupby("entity", group_keys=True).apply(proxy_nps, include_groups=False).reset_index()
nps_by_entity

,entity,n_reviews,pct_promoter,pct_passive,pct_detractor,proxy_nps
0,HDFC Bank,625.0,44.2,5.6,50.2,-6.1
1,Jupiter,677.0,21.9,3.4,74.7,-52.9


## Also break out by platform

Worth carrying forward: HDFC and Jupiter both show a wide App Store vs. Google Play split
(see Week 3 `sentiment_summary.csv`) — the pooled proxy-NPS below smooths over that.

In [3]:
nps_by_platform = (
    df.groupby(["entity", "platform"], group_keys=True)
    .apply(proxy_nps, include_groups=False)
    .reset_index()
)
nps_by_platform

,entity,platform,n_reviews,pct_promoter,pct_passive,pct_detractor,proxy_nps
0,HDFC Bank,app_store,448.0,52.0,6.2,41.7,10.3
1,HDFC Bank,google_play,177.0,24.3,4.0,71.8,-47.5
2,Jupiter,app_store,495.0,15.6,3.6,80.8,-65.3
3,Jupiter,google_play,182.0,39.0,2.7,58.2,-19.2


In [4]:
nps_by_entity.to_csv(PROCESSED / "proxy_nps.csv", index=False)
nps_by_platform.to_csv(PROCESSED / "proxy_nps_by_platform.csv", index=False)
print("-> data/processed/proxy_nps.csv")
print("-> data/processed/proxy_nps_by_platform.csv")

-> data/processed/proxy_nps.csv
-> data/processed/proxy_nps_by_platform.csv
